In [ ]:
# ---------------------------------------------------------------------------
# Configuration flags — toggle before running the notebook
# ---------------------------------------------------------------------------
ENABLE_TRACING = False  # Set to True to enable Phoenix tracing


In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=LfhaYQFIrkAA69200RLAsAsUqzHbau&access_type=offline&code_challenge=zosnCY60-FWjgVlPBioXj9wriEi1d_fVprgiT6RNVuI&code_challenge_method=S256


Credentials saved to file: [C:\Users\L137844\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "dev-mq-tech-transfer" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import google.auth

_, project_id = google.auth.default()
print(f"Project ID: {project_id}")

Project ID: dev-mq-tech-transfer


In [ ]:
# Instrumentation — run once per kernel session
if ENABLE_TRACING:
    from phoenix.otel import register
    from openinference.instrumentation.google_adk import GoogleADKInstrumentor

    tracer_provider = register(
        project_name="gap_assessment",
        endpoint="http://localhost:6006/v1/traces"
    )
    GoogleADKInstrumentor().instrument(tracer_provider=tracer_provider)
    print("Phoenix tracing enabled.")
else:
    tracer_provider = None
    print("Tracing disabled.")


c:\Users\L137844\Downloads\gap_assesment\tredence_gap_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenTelemetry Tracing Details
|  Phoenix Project: gap_assessment
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [4]:
import json
import os
from typing import Any, Dict, List, Optional

import vertexai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.api_core.client_options import ClientOptions
from google.cloud import logging as gcp_logging
from google.cloud import discoveryengine_v1 as discoveryengine
from google.genai import types as genai_types
from vertexai.generative_models import GenerativeModel, GenerationConfig

<!-- ## Chunk-Based RAG Workflow -->

In [ ]:

# ---------------------------------------------------------------------------
# System prompt
# ---------------------------------------------------------------------------

CONVERSATIONAL_PROMPT = """\
You are a pharmaceutical technology-transfer analyst. Your job is to reconstruct the
FACILITY section of a Technology-Transfer Gap Assessment by retrieving from and reasoning
over a document datastore, then emitting a single structured JSON object.

## What a gap assessment captures
For the evaluated area, compare four things:
- sending_site_requirements: product/process characteristics inherited from the sending
  process that the receiving facility must satisfy.
- global_requirements: corporate, pharmacopeial, and regulatory standards the facility must
  meet (e.g., EU GMP Annex 1, internal global quality standards, USP chapters, internal guidance).
- receiving_site_evidence: what the receiving site has designed, installed, qualified, or
  otherwise demonstrated.
- gaps: requirements not yet satisfied by evidence — missing, pending, or not-yet-available —
  and risks introduced by the receiving context that require controls not yet verified.

## Retrieval strategy (this corpus is deliberately hard to retrieve from)
Do NOT rely on one broad query. Decompose the Facility section into atomic facts, issue a
separate targeted retrieval query for each, then synthesize.

1. One fact per query. Cover at minimum: design standards & capacity; room classification
   (grades); air-pressurization / room pressure differentials (query EACH boundary separately —
   isolator↔surrounding room, air-lock↔room, grade↔grade, room↔corridor, containment enclosure);
   personnel/material/waste/equipment flows; raw-material & product handling and dispensing;
   environmental temperature (query EACH context separately: formulation WFI, general room
   ambient, cold storage/freezer, EM incubation); environmental humidity (general-room band and
   the dispensing limit are SEPARATE); cross-contamination / product mix-up; shared-building /
   multi-product context; utilities (WFI, clean steam, process air); cleaning & sanitising agents;
   isolator bio-decontamination (the decontaminant WORKING CONCENTRATION and the RESIDUAL
   PRODUCT-EXPOSURE LIMIT are different quantities — query each); cold storage / freezer; overall
   qualification status and milestones.

2. Rotate vocabulary. The same concept is worded differently across documents; if a query
   returns little, re-issue with synonyms:
     humidity ↔ "relative humidity" / "moisture level" / "HR"
     cross-contamination ↔ "product carryover" / "cross contamination"
     pressure ↔ "air-pressurization scheme" / "inter-room pressure regime" / "room pressure differentials"
     hydrogen peroxide ↔ "VPHP" / "vapour-phase hydrogen peroxide" / "vaporized hydrogen peroxide"
   A value may be STATED in one document and only REFERRED TO (without the number) in another.
   Trust the document that states the actual value; a pointer without a value is not the answer.

3. Name the context for any quantity that has several context-specific values (temperature,
   humidity, differential pressure). A bare "what is the temperature/humidity/pressure" is
   ambiguous and MUST be split by context or boundary.

4. Reassemble split tables. A parameter table may be chunked so column headers and data rows
   are retrieved separately (one chunk has labels but no numbers; another has numbers but no
   labels). Retrieve both and rejoin by position.

5. Follow multi-hop chains. A single topic may span documents: a hazard stated in one, the
   control in another, the verification in a third. Retrieve all links before concluding
   (e.g., cross-contamination: co-location hazard → segregated flows + pressure cascade control
   → airflow-visualisation / EM / pressure-differential verification).

6. Run negative-control queries. For anything that seems like it should exist, confirm the
   datastore actually contains it. Documents may be CITED but ABSENT, or a detail may be
   explicitly deferred to a report that is out of scope. These are NOT evidence.

## Classification and anti-fabrication rules
- Sort each retrieved fact into requirement (sending or global), evidence, or gap.
- A requirement with no corresponding satisfied evidence — because it is missing, "in progress",
  "ongoing", scheduled for later (e.g., "before APS", "before drug-substance reception"), or
  deferred to a document not in the datastore — is a GAP. Never invent a passing result.
- Never fabricate values, owners, dates, or evidence. If the datastore does not state something,
  omit it (or use null for gap owner/due_date). Do NOT present a cited-but-absent document or an
  out-of-scope pointer as evidence.
- Preserve every masked/placeholder token EXACTLY as written (e.g., [COMPOUND_1],
  [RECEIVING_SITE_1], [DOC_ID_216], [PERSON_7]). Do not expand, guess, or unmask them.
- Report figures exactly as stated, with units and qualifiers ("NMT", "≤", registered ranges).
  Do not round or normalize.

## Sourcing
- Every requirement and evidence entry carries a "source" citing the most specific locator
  available: the document identifier and, if determinable, the section/heading
  (e.g., "[DOC_ID_NEW_3] §9 Dispensing Suite Qualification"). If a fact is synthesized across
  documents (multi-hop), cite all contributing documents in the source string.

## CQAs
- Populate cqa_at_risk only with critical quality attributes the datastore associates with that
  hazard (e.g., product identity; purity & product-related impurities; particulate matter). Use
  the corpus's own wording; do not add CQAs the corpus does not tie to the gap.

## Output contract
- Output EXACTLY ONE JSON object and NOTHING else: no prose, no explanation, no markdown fences.
- The object must have exactly these keys and shapes:

{
  "facility_area": "",
  "sending_site_requirements": [{"requirement": "", "source": ""}],
  "global_requirements": [{"requirement": "", "source": ""}],
  "receiving_site_evidence": [{"evidence": "", "source": ""}],
  "gaps": [{"gap_description": "", "risk_to_product_quality": "", "cqa_at_risk": [], "owner": "", "due_date": ""}]
}

- "facility_area": the evaluated-area label as it appears in the source; use "Facility" if
  unspecified.
- Each list contains one entry per atomic fact; be comprehensive across all Facility sub-topics
  listed above.
- In gaps, "owner" and "due_date" are null unless the datastore states them; "due_date" may be a
  milestone phrase (e.g., "before APS") only if the datastore ties remediation to it.
- "cqa_at_risk" is an array of strings (empty array if none stated).
"""


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _log_step(
    project_id: str, step: str, user_id: str, query: str, response: str, citation: Any
) -> None:
    payload = {
        "Step": step,
        "User_id": user_id,
        "User_query": query,
        "Response": response,
        "Citation": citation,
    }
    gcp_logging.Client(project=project_id).logger("rag_workflow").log_struct(
        payload, severity="INFO"
    )


def _chunk_search_passages(
    query: str,
    project_id: str,
    location: str,
    datastore_id: str,
    page_size: int = 8,
) -> List[Dict[str, Any]]:
    """Chunk-mode search returning passage dicts."""
    endpoint = (
        "discoveryengine.googleapis.com"
        if location.lower() == "global"
        else f"{location}-discoveryengine.googleapis.com"
    )
    serving_config = (
        f"projects/{project_id}/locations/{location}/collections/default_collection/"
        f"dataStores/{datastore_id}/servingConfigs/default_config"
    )
    spec = discoveryengine.SearchRequest.ContentSearchSpec(
        search_result_mode=discoveryengine.SearchRequest.ContentSearchSpec.SearchResultMode.CHUNKS,
    )
    req = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=page_size,
        content_search_spec=spec,
    )

    passages: List[Dict[str, Any]] = []
    client = discoveryengine.SearchServiceClient(
        client_options=ClientOptions(api_endpoint=endpoint)
    )
    for r in client.search(req).results:
        chunk = getattr(r, "chunk", None)
        if chunk is None:
            continue

        content = str(getattr(chunk, "content", "") or "").strip()
        if not content:
            continue

        doc_meta = getattr(chunk, "document_metadata", None)
        title = str(getattr(doc_meta, "title", "") or "") if doc_meta else ""
        uri = str(getattr(doc_meta, "uri", "") or "") if doc_meta else ""

        chunk_name = str(getattr(chunk, "name", "") or "")
        parts = chunk_name.split("/")
        doc_id = parts[-3] if len(parts) >= 3 else ""

        passages.append(
            {"doc_id": doc_id, "title": title, "uri": uri, "content": content}
        )
    return passages


def make_search_tool(project_id: str, location: str, datastore_id: str):
    """Return an ADK-compatible search tool."""

    def search_datastore(query: str) -> str:
        """Search the document datastore for relevant chunks matching the query.

        Args:
            query: The search query to find relevant document chunks.

        Returns:
            Formatted document chunks with citation IDs, titles, URIs, and content.
        """
        all_passages = _chunk_search_passages(query, project_id, location, datastore_id)

        if not all_passages:
            return "No relevant documents found."

        lines = []
        for idx, p in enumerate(all_passages, 1):
            lines.append(
                f"[C{idx}]\n"
                f"title: {p['title']}\n"
                f"uri: {p['uri']}\n"
                f"content: {p['content']}"
            )
        return "\n\n".join(lines)

    return search_datastore


# ---------------------------------------------------------------------------
# Agent runner
# ---------------------------------------------------------------------------

async def run_rag_agent(
    query: str,
    user_id: str,
    project_id: str,
    location: str,
    datastore_id: str,
    model_location: str = "us-central1",
) -> str:
    vertexai.init(project=project_id, location=model_location)
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
    os.environ["GOOGLE_CLOUD_LOCATION"] = model_location

    search_tool = make_search_tool(project_id, location, datastore_id)

    agent = Agent(
        model="gemini-2.5-flash",
        name="rag_agent",
        instruction=CONVERSATIONAL_PROMPT,
        tools=[search_tool],
    )

    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name="rag_app", user_id=user_id)
    runner = Runner(agent=agent, app_name="rag_app", session_service=session_service)

    _log_step(project_id, "agent_start", user_id, query, "", [])

    result_text = ""
    user_message = genai_types.Content(
        role="user",
        parts=[genai_types.Part.from_text(text=query)],
    )
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=user_message,
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        result_text = part.text
                        break

    _log_step(project_id, "agent_complete", user_id, query, result_text, [])

    return result_text


<!-- ## Sample Run -->


In [8]:
SAMPLE_QUERY = "tell me about the gaps in facilities"
# TARGET_DATASTORE_ID = "masked-data_1781603945004"

TARGET_DATASTORE_ID = "tredence-generated-docs_1782905290782"

result = await run_rag_agent(
    query=SAMPLE_QUERY,
    user_id="sample-user-001",
    project_id=project_id,
    location="us",
    datastore_id=TARGET_DATASTORE_ID,
)

print(result)

Transient error HTTPConnectionPool(host='localhost', port=6006): Max retries exceeded with url: /v1/traces (Caused by NewConnectionError("HTTPConnection(host='localhost', port=6006): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it")) encountered while exporting span batch, retrying in 1.05s.
Failed to export span batch due to timeout, max retries or shutdown.
Transient error HTTPConnectionPool(host='localhost', port=6006): Max retries exceeded with url: /v1/traces (Caused by NewConnectionError("HTTPConnection(host='localhost', port=6006): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it")) encountered while exporting span batch, retrying in 1.10s.
Failed to export span batch due to timeout, max retries or shutdown.
Transient error HTTPConnectionPool(host='localhost', port=6006): Max retries exceeded with url: /v1/traces (Caus

```json
[
  {
    "facility_area": "Facility design, capacity, and room classifications (including dedicated line status, aseptic design meeting EU Annex 1 and GQS202)",
    "sending_site_requirements": [
      {
        "requirement": "Facility design should mitigate cross contamination and product mix-up hazards, potentially through line dedication.",
        "source": "[C1] Conclusion"
      }
    ],
    "global_requirements": [],
    "receiving_site_evidence": [
      {
        "evidence": "New, product-dedicated cartridge line established at [RECEIVING_SITE_1] in Building B700.",
        "source": "[C2] Abstract"
      },
      {
        "evidence": "Design intent for facility layout and capacity, and room classification scheme.",
        "source": "[C2] Abstract"
      },
      {
        "evidence": "Cleanroom-grade design is governed by the C&Q summary.",
        "source": "[C3] Abstract"
      }
    ],
    "gaps": []
  },
  {
    "facility_area": "Air pressurization schemes and